# Controlled CKD experiments — PyTorch

Run the setup and data checks first. Every training, search, plotting, and final-test stage is explicitly gated below. **All gates default to False:** Run All does not train CKD models or inspect final-test performance.

CKD=1. CV uses threshold 0.5; validation selects maximum F2 with precision ≥0.80 (ties: precision, then threshold). The 80-row test partition is accessed only by the final-test stage.

The prior EDA examined all records. Source repairs and absent patient/timing metadata limit claims of independence. This is a machine-learning benchmark, not a clinical diagnosis tool.


In [2]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Open this notebook from the project directory")
sys.path.insert(0, str(ROOT))
import torch
import pandas as pd
from src.data import prepare_splits
from src.train import run_experiment, search_configs, loss_configs
from src.plots import compare_runs, permutation_importance, plot_predictions
from src.final_test import freeze_finalist, evaluate_final, report_saved_test
DATA = ROOT / "Chronic_Kidney_Disease/chronic_kidney_disease_full.csv"
SPLITS = ROOT / "data/splits"
RESULTS = ROOT / "results"
FINAL = RESULTS / "final_test"
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)


## 1. Reuse the frozen split and inspection hand-off
No EDA or source conversion is rerun. Metadata verifies file and split identity.

In [3]:
manifest = prepare_splits(DATA, SPLITS)
print({key: len(manifest[key]) for key in ("train", "validation", "test")})
print("Dataset:", manifest["dataset_hash"])
print("Split:", manifest["split_id"])
print((ROOT / "data/inspection_handoff.md").read_text())


{'train': 240, 'validation': 80, 'test': 80}
Dataset: c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e47bcfbfaf3fe96b5fd
Split: 3bbbe0819585c309e9a8237eded582c758ca8b55e0987dd7aa881028c844c55e
# Inspection hand-off

Source: existing data_exploration_notbook.ipynb, cells 2–8 and 10–23, and local ARFF metadata.

- Dataset SHA-256: `c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e47bcfbfaf3fe96b5fd`; 400 rows, 25 columns.
- Target: class; CKD=1 (250), non-CKD=0 (150).
- Numerical predictors: age, bp, bgr, bu, sc, sod, pot, hemo, pcv, wbcc, rbcc.
- Ordered predictors: sg, al, su. Other categories are binary with mappings in schema.json.
- Existing inspection found no exact or predictor-only duplicates. No patient IDs or timestamps are present; repeated-patient independence cannot be verified.
- Existing missingness: rbc 38%, rbcc 32.75%, wbcc 26.5%, pot 22%, sod 21.75%; full per-feature counts are in validation_rules.json.
- Source converter removed one extra empty field on three ARFF rows. Th

## 2. Inspect the predeclared experiment menu
E01 is the reference for every first-pass ablation. E06 full-with-labs is the same configuration as E01; history-only excludes `ane` because ascertainment is uncertain. E07 StandardScaler is already E01. No need to run identical references twice.

In [4]:
CONFIGS = json.loads((ROOT / "configs/experiments.json").read_text())
BY_ID = {c["experiment_id"]: c for c in CONFIGS}
pd.DataFrame([{ "id": c["experiment_id"], "model": c["model"], "change": c.get("preprocessing", {})} for c in CONFIGS])


,id,model,change
0,E00-most_frequent,dummy,{}
1,E00-always_ckd,dummy,{}
2,E01-lr,lr,{}
3,E02-lr,lr,{'indicators': True}
4,E03-lr,lr,{'imputer': 'knn'}
5,E04-lr,lr,{'imputer': 'iterative'}
6,E05-30-lr,lr,{'missing_threshold': 0.3}
7,E05-50-lr,lr,{'missing_threshold': 0.5}
8,E06-history-lr,lr,{'feature_set': 'history'}
9,E06-redundant-lr,lr,{'selection': 'redundant'}


## 3. E00/E01 baselines
Set the gate to True when you want to train. Each run saves CV scores, held-out predictions, fitted pipelines, parameters, seeds, feature names, convergence warnings, environment and code provenance. Repeating an existing run is rejected to protect artifacts.

In [5]:
RUN_BASELINES = False
baseline_ids = [c["experiment_id"] for c in CONFIGS if c["experiment_id"].startswith(("E00", "E01"))]
if RUN_BASELINES:
    for experiment_id in baseline_ids:
        print(f"Starting {experiment_id}...", flush=True)
        run_dir = run_experiment(BY_ID[experiment_id], DATA, SPLITS, RESULTS)
        print(f"Saved: {run_dir}", flush=True)
        display(pd.DataFrame([json.loads((run_dir / "summary.json").read_text())]))
else:
    print("Skipped: RUN_BASELINES=False. Set it to True and rerun this cell when ready.")


Skipped: RUN_BASELINES=False. Set it to True and rerun this cell when ready.


## 4. E02–E08 controlled ablations
Run within a model family first. Missingness filtering, imputation, scaling, redundancy removal and selection are fitted separately inside every training fold. A failed fit leaves a failure record instead of being silently dropped.

In [6]:
RUN_ABLATIONS = True
ablation_ids = [c["experiment_id"] for c in CONFIGS if not c["experiment_id"].startswith(("E00", "E01"))]
if RUN_ABLATIONS:
    for experiment_id in ablation_ids:
        print(f"Starting {experiment_id}...", flush=True)
        run_dir = run_experiment(BY_ID[experiment_id], DATA, SPLITS, RESULTS)
        print(f"Saved: {run_dir}", flush=True)
        display(pd.DataFrame([json.loads((run_dir / "summary.json").read_text())]))
else:
    print("Skipped: RUN_ABLATIONS=False. Set it to True and rerun this cell when ready.")


Starting E02-lr...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E02-lr-9749dc0a60e3


,run_id,threshold,eligible,seconds,validation,cv_f2_mean,status
0,E02-lr-9749dc0a60e3,0.695664,True,0.89035,"{'tp': 50, 'fp': 0, 'tn': 30, 'fn': 0, 'precis...",0.994631,complete


Starting E03-lr...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E03-lr-da1bfe7c813c


,run_id,threshold,eligible,seconds,validation,cv_f2_mean,status
0,E03-lr-da1bfe7c813c,0.162501,True,0.712531,"{'tp': 50, 'fp': 1, 'tn': 29, 'fn': 0, 'precis...",0.978451,complete


Starting E04-lr...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E04-lr-433fb9c4cde3


,run_id,threshold,eligible,seconds,validation,cv_f2_mean,status
0,E04-lr-433fb9c4cde3,0.200197,True,0.829927,"{'tp': 50, 'fp': 0, 'tn': 30, 'fn': 0, 'precis...",0.983893,complete


... (64 outputs left out)

Starting E07-robust-nn...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E07-robust-nn-0e69a0ff63d9


,run_id,threshold,eligible,seconds,validation,cv_f2_mean,status
0,E07-robust-nn-0e69a0ff63d9,0.824966,True,14.921409,"{'tp': 50, 'fp': 0, 'tn': 30, 'fn': 0, 'precis...",0.989189,complete


Starting E07-minmax-nn...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E07-minmax-nn-635841702b91


,run_id,threshold,eligible,seconds,validation,cv_f2_mean,status
0,E07-minmax-nn-635841702b91,0.30041,True,17.695691,"{'tp': 50, 'fp': 0, 'tn': 30, 'fn': 0, 'precis...",0.969357,complete


## 5. Review saved development evidence
Seed-averaged fold scores describe fold variability; `runs.csv` retains each seed separately. Paired deltas share folds. Small differences do not establish superiority. Validation performance is selected/tuned evidence, not independent test performance.

In [7]:
runs = sorted(p.parent for p in RESULTS.glob("*/summary.json") if json.loads(p.read_text()).get("status") == "complete") if RESULTS.exists() else []
if (RESULTS / "runs.csv").exists():
    display(pd.read_csv(RESULTS / "runs.csv"))
MAKE_PLOTS = False
if MAKE_PLOTS and runs:
    compare_runs(runs, RESULTS / "figures")
if not runs:
    print("No completed experiments yet. Enable RUN_BASELINES in section 3 and run that cell first.")
elif not MAKE_PLOTS:
    print("Plots skipped: set MAKE_PLOTS=True to generate figures.")


,experiment_id,run_id,split_id,dataset_hash,feature_set,imputer,encoder,scaler,model,model_params,...,val_fp,val_tn,val_fn,val_precision,val_recall,val_specificity,val_f2,val_brier,val_roc_auc,val_ap
0,E00-always_ckd,E00-always_ckd-3f45580925ce,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,none,dummy,"{""strategy"": ""always_ckd""}",...,30,0,0,0.625,1.0,0.0,0.892857,0.375000,0.5,0.625
1,E00-most_frequent,E00-most_frequent-d31a8fa145f2,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,none,dummy,"{""strategy"": ""most_frequent""}",...,30,0,0,0.625,1.0,0.0,0.892857,0.375000,0.5,0.625
2,E01-lr,E01-lr-a3ee046c414d,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,standard,lr,{},...,0,30,0,1.000,1.0,1.0,1.000000,0.007880,1.0,1.000
3,E01-nn,E01-nn-fd199c5137ed,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,standard,nn,{},...,0,30,0,1.000,1.0,1.0,1.000000,0.001433,1.0,1.000
4,E01-nn,E01-nn-fd199c5137ed,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,standard,nn,{},...,0,30,0,1.000,1.0,1.0,1.000000,0.002861,1.0,1.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,E08-lr,E08-lr-e2c9444e9d8b,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,standard,lr,{},...,0,30,0,1.000,1.0,1.0,1.000000,0.005028,1.0,1.000
80,E08-nn,E08-nn-778d7a318e62,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,standard,nn,{},...,0,30,0,1.000,1.0,1.0,1.000000,0.000913,1.0,1.000
81,E08-nn,E08-nn-778d7a318e62,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,standard,nn,{},...,0,30,0,1.000,1.0,1.0,1.000000,0.000783,1.0,1.000
82,E08-nn,E08-nn-778d7a318e62,3bbbe0819585c309e9a8237eded582c758ca8b55e0987d...,c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e...,all,median,schema,standard,nn,{},...,0,30,0,1.000,1.0,1.0,1.000000,0.000720,1.0,1.000


Plots skipped: set MAKE_PLOTS=True to generate figures.


## 6. E09 training-CV searches
Enter configuration IDs supported by CV evidence. The search runs 8 LR, 24 sampled RF, or 8 NN settings; it never uses test labels. Choose a finalist using CV stability and validation evidence, not a single lucky seed.

In [19]:
RUN_SEARCH = True
search_parent_ids = ["E03-nn"]  # e.g. ["E01-lr", "E02-rf", "E01-nn"]
if RUN_SEARCH:
    if not search_parent_ids:
        raise ValueError("Choose parent configurations from your CV evidence")
    for experiment_id in search_parent_ids:
        for config in search_configs(BY_ID[experiment_id]):
            print(f"Starting {config['experiment_id']}...", flush=True)
            run_dir = run_experiment(config, DATA, SPLITS, RESULTS)
            print(f"Saved: {run_dir}", flush=True)
else:
    print("Skipped: RUN_SEARCH=False. Set it to True and rerun this cell when ready.")


Starting E09-nn-00...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E09-nn-00-5267b50fcbd5
Starting E09-nn-01...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E09-nn-01-0b32164966ae
Starting E09-nn-02...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E09-nn-02-0c3bbcd429fd
Starting E09-nn-03...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E09-nn-03-58b563b362eb
Starting E09-nn-04...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E09-nn-04-5f3ba7503145
Starting E09-nn-05...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E09-nn-05-8331d86e4b9a
Starting E09-nn-06...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E09-nn-06-bb9ad1f4c2ff
Starting E09-nn-07...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E09-nn-

## 7. E10 controlled NN loss study
Choose one fixed NN configuration (or load a tuned run’s `config.json`). All four losses share features, architecture, optimizer, seed set and stopping policy. Raw loss values are not comparable between losses.

In [11]:
RUN_LOSSES = True
loss_parent = BY_ID["E01-nn"]
if RUN_LOSSES:
    for config in loss_configs(loss_parent):
        print(f"Starting {config['experiment_id']}...", flush=True)
        run_dir = run_experiment(config, DATA, SPLITS, RESULTS)
        print(f"Saved: {run_dir}", flush=True)
else:
    print("Skipped: RUN_LOSSES=False. Set it to True and rerun this cell when ready.")


Starting E10-bce...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E10-bce-95033bdbf830
Starting E10-weighted_bce...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E10-weighted_bce-de91bb47ff22
Starting E10-mse...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E10-mse-dfc01ef0830a
Starting E10-focal...
Saved: /Users/nguyencongkhanh/Downloads/chronic-kidney-disease-prediction/results/E10-focal-300e39803091


## 8. Inspect a candidate, then freeze it
Choose an eligible saved run. NN predictions average the three fixed seeds. Permutation importance uses validation rows and a fixed selected threshold. Freezing copies the fitted pipelines without refitting on validation. An existing frozen selection cannot be replaced through this API.

In [14]:
selected_run = "results/E10-bce-95033bdbf830"  # Path to a completed results/<run_id> directory
MAKE_IMPORTANCE = True
FREEZE_FINALIST = True
if MAKE_IMPORTANCE:
    if selected_run is None: raise ValueError("Select a completed run")
    permutation_importance(selected_run, DATA, SPLITS)
if FREEZE_FINALIST:
    if selected_run is None: raise ValueError("Select a completed eligible run")
    freeze_finalist(selected_run, FINAL)
if not MAKE_IMPORTANCE and not FREEZE_FINALIST:
    print("Selection stage skipped. Choose a completed run after reviewing development results.")


## 9. E11 — final test, once
Only enable this after selecting and freezing the winner. The call checks dataset, split, model and source hashes, then creates an exclusive test-start marker. Do not iterate on test results. If reporting fails after predictions were saved, regenerate reports below; do not repeat inference.

In [15]:
RUN_FINAL_TEST_ONCE = True
if RUN_FINAL_TEST_ONCE:
    final_report = evaluate_final(DATA, SPLITS, FINAL)
    display(final_report)
else:
    print("Skipped: RUN_FINAL_TEST_ONCE=False. Set it to True and rerun this cell when ready.")


{'selection': {'run_id': 'E10-bce-95033bdbf830',
  'pipeline_hash': '99bcfe594ce12a720c9deacb2473518138618a6217cd3ef4f7a0c0e05a2491ef',
  'threshold': 0.643125575372931,
  'dataset_hash': 'c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e47bcfbfaf3fe96b5fd',
  'split_id': '3bbbe0819585c309e9a8237eded582c758ca8b55e0987dd7aa881028c844c55e',
  'code_hash': 'bc1a3face24e6d16116526b3f3de8a16beb7bc4f35f48c5629b6ef69b9ef8246'},
 'metrics': {'tp': 50,
  'fp': 0,
  'tn': 30,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'specificity': 1.0,
  'f2': 1.0,
  'brier': 0.00010992229634466527,
  'roc_auc': 1.0,
  'ap': 1.0},
 'intervals': {'f2': [1.0, 1.0],
  'precision': [1.0, 1.0],
  'recall': [1.0, 1.0],
  'specificity': [1.0, 1.0],
  'ap': [1.0, 1.0],
  'roc_auc': [1.0, 1.0],
  'brier': [6.766857118606491e-06, 0.00030069624695275174]},
 'n_resamples': 2000,
 'limitations': ['Full dataset examined in prior EDA',
  'No verified patient or temporal identifiers',
  'Three unverified ARFF repairs',
  

## 10. Rebuild the final report from saved predictions
This reads already-scored test predictions without refitting or retuning. Confidence intervals condition on the chosen pipeline and class counts; they exclude model-selection uncertainty.

In [16]:
REPORT_SAVED_TEST = True
if REPORT_SAVED_TEST:
    report = report_saved_test(FINAL)
    rows = json.loads((FINAL / "predictions.json").read_text())
    plot_predictions([r["y"] for r in rows], [r["probability"] for r in rows], report["selection"]["threshold"], FINAL / "figures")
    display(report)
else:
    print("Skipped: REPORT_SAVED_TEST=False. Set it to True and rerun this cell when ready.")


{'selection': {'run_id': 'E10-bce-95033bdbf830',
  'pipeline_hash': '99bcfe594ce12a720c9deacb2473518138618a6217cd3ef4f7a0c0e05a2491ef',
  'threshold': 0.643125575372931,
  'dataset_hash': 'c0eaa5003c90a82d9254be8bc2f9d30701956322a3ff9e47bcfbfaf3fe96b5fd',
  'split_id': '3bbbe0819585c309e9a8237eded582c758ca8b55e0987dd7aa881028c844c55e',
  'code_hash': 'bc1a3face24e6d16116526b3f3de8a16beb7bc4f35f48c5629b6ef69b9ef8246'},
 'metrics': {'tp': 50,
  'fp': 0,
  'tn': 30,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'specificity': 1.0,
  'f2': 1.0,
  'brier': 0.00010992229634466527,
  'roc_auc': 1.0,
  'ap': 1.0},
 'intervals': {'f2': [1.0, 1.0],
  'precision': [1.0, 1.0],
  'recall': [1.0, 1.0],
  'specificity': [1.0, 1.0],
  'ap': [1.0, 1.0],
  'roc_auc': [1.0, 1.0],
  'brier': [6.766857118606491e-06, 0.00030069624695275174]},
 'n_resamples': 2000,
 'limitations': ['Full dataset examined in prior EDA',
  'No verified patient or temporal identifiers',
  'Three unverified ARFF repairs',
  

## 11. Your analysis

Use `report_template.md` in the project root as a starting point.

1. Which missing-value treatment improved paired CV F2, and at what specificity?
2. Did scaling, log features or selection justify their complexity?
3. How did LR, RF and the three-seed NN compare?
4. Did changing NN loss help consistently across the same folds and seeds?
5. What was the incremental value of laboratory features?
6. Were differences larger than fold/seed variability, and how do dataset limitations affect interpretation?

Report failures, infeasible precision constraints, actual FP/FN counts, and the always-CKD baseline. Do not claim the test set was untouched by prior EDA.
